In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set clean aesthetic for presentation
sns.set_theme(style="whitegrid")

In [ ]:
# Load your final CSV dataset
df = pd.read_csv("final_dataset.csv")

# MAGIC LINE: Combine 1 and 2 into 1, keep 0 as 0
df['fire_label'] = df['fire_label'].map({0: 0, 1: 1, 2: 1})

print("Dataset transformed to Binary Classifier!")
print("New Class Counts:\n", df['fire_label'].value_counts())

# Class Distribution Analysis
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='fire_label', palette='Set1')
plt.title('Binary Class Distribution')
plt.xlabel('Fire Risk Level (0=Low, 1=Risk/High)')
plt.ylabel('Total Samples')
plt.show()

In [ ]:
# Select your physical atmospheric features
features = ['temp_max_c', 'humidity_pct', 'rain_binary']

plt.figure(figsize=(14, 4))
for i, col in enumerate(features, 1):
    plt.subplot(1, 3, i)
    # Using specific colors matching your earlier dashboard layouts
    sns.boxplot(data=df, y=col, color='#2980b9' if i==1 else '#e67e22' if i==2 else '#27ae60')
    plt.title(f'Distribution & Outliers: {col}')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
# Matrix correlation for your continuous and target parameters
corr_matrix = df[['temp_max_c', 'humidity_pct', 'rain_binary', 'fire_label']].corr()

# Plotting the matrix with explicit formatting
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=1)
plt.title('System Feature Correlation Matrix')
plt.show()

In [ ]:
# Extract feature and target matrices
X = df[['temp_max_c', 'humidity_pct', 'rain_binary']]
y = df['fire_label']

# Stratified splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize Binary Classifier
rf_model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')

# Fit model parameters
rf_model.fit(X_train, y_train)

In [ ]:
import joblib

# Save the trained model to a file
joblib.dump(rf_model, 'forest_fire_model.pkl')

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"=========================================")
print(f"ACCURACY: {accuracy * 100:.2f}%")
print(f"=========================================\n")

# Complete performance details - FORCED BINARY LABELS
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['Low Risk (0)', 'High Risk/Alert (1)']))

# Confusion matrix setup for 2 categories - FORCED BINARY LABELS
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Predicted Low (0)', 'Predicted Risk (1)'], 
            yticklabels=['Actual Low (0)', 'Actual Risk (1)'])
plt.title('Binary Confusion Matrix')
plt.ylabel('True State')
plt.xlabel('Model Assigned State')
plt.show()